# 02 — EDA do Clima no Estado de São Paulo

Este notebook realiza a Análise Exploratória de Dados (EDA) do dataset meteorológico horário já recortado para o **estado de São Paulo (SP)**.

A proposta desta EDA é construir uma narrativa climática para apoiar o projeto de Big Data:

- entender a qualidade dos dados;
- analisar a distribuição espacial das estações meteorológicas;
- comparar regiões do estado de São Paulo;
- observar diferenças entre litoral, serra, áreas urbanas e interior;
- investigar relações entre temperatura, umidade, precipitação, altitude e localização;
- preparar os principais insights que justificam a etapa de modelagem preditiva.

O objetivo final do projeto é prever:

- a temperatura de amanhã;
- a temperatura média da próxima semana.

## 1. Inicialização da SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EDA_Clima_SP_SparkSQL")
    .getOrCreate()
)

spark

## 2. Importação das bibliotecas

In [ ]:
import unicodedata
import re

import plotly.express as px
import plotly.graph_objects as go

## 3. Carregamento da base Parquet

A base utilizada neste notebook já está em formato Parquet e já foi previamente filtrada para o estado de São Paulo.

Por isso, as próximas análises consideram diretamente o recorte paulista, sem necessidade de novo filtro por estado.

In [ ]:
parquet_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df_raw = spark.read.parquet(parquet_path)

df_raw.createOrReplaceTempView("weather_raw")

spark.sql("""
    SELECT *
    FROM weather_raw
    LIMIT 5
""").show(truncate=False)

## 4. Estrutura inicial do dataset

In [ ]:
df_raw.printSchema()

print(f"Quantidade de colunas: {len(df_raw.columns)}")

print("\nColunas originais:")
for c in df_raw.columns:
    print("-", c)

## 5. Normalização dos nomes das colunas

In [ ]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df_raw.columns]
df = df_raw.toDF(*colunas_normalizadas)

df.columns

In [ ]:
df.createOrReplaceTempView("weather_normalizado")

spark.sql("""
    SELECT *
    FROM weather_normalizado
    LIMIT 5
""").show(truncate=False)

## 6. Renomeação das variáveis principais

In [ ]:
mapa_renomeacao = {
    "precipitacao_total_horario_mm": "precipitacao",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb": "pressao",
    "pressao_atmosferica_max_na_hora_ant_aut_mb": "pressao_maxima",
    "pressao_atmosferica_min_na_hora_ant_aut_mb": "pressao_minima",
    "radiacao_global_kj_m2": "radiacao",
    "temperatura_do_ar_bulbo_seco_horaria_c": "temperatura",
    "temperatura_maxima_na_hora_ant_aut_c": "temperatura_maxima",
    "temperatura_minima_na_hora_ant_aut_c": "temperatura_minima",
    "umidade_relativa_do_ar_horaria": "umidade",
    "umidade_rel_max_na_hora_ant_aut": "umidade_maxima",
    "umidade_rel_min_na_hora_ant_aut": "umidade_minima",
    "vento_direcao_horaria_gr_gr": "direcao_vento",
    "vento_rajada_maxima_m_s": "rajada_vento",
    "vento_velocidade_horaria_m_s": "velocidade_vento",
    "height": "altitude"
}

df.createOrReplaceTempView("weather_normalizado")

colunas_select = []

for coluna in df.columns:
    if coluna in mapa_renomeacao:
        colunas_select.append(f"`{coluna}` AS `{mapa_renomeacao[coluna]}`")
    else:
        colunas_select.append(f"`{coluna}`")

query_renomeacao = f"""
    SELECT
        {", ".join(colunas_select)}
    FROM weather_normalizado
"""

df = spark.sql(query_renomeacao)

df.createOrReplaceTempView("weather_renomeado")

spark.sql("""
    DESCRIBE weather_renomeado
""").show(100, truncate=False)

## 7. Conversão das variáveis numéricas

In [ ]:
colunas_numericas = [
    "latitude",
    "longitude",
    "altitude",
    "temperatura",
    "temperatura_maxima",
    "temperatura_minima",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

colunas_numericas = [c for c in colunas_numericas if c in df.columns]

df.createOrReplaceTempView("weather_renomeado")

colunas_select = []

for coluna in df.columns:
    if coluna in colunas_numericas:
        colunas_select.append(
            f"CAST(REPLACE(CAST(`{coluna}` AS STRING), ',', '.') AS DOUBLE) AS `{coluna}`"
        )
    else:
        colunas_select.append(f"`{coluna}`")

query_conversao_numerica = f"""
    SELECT
        {", ".join(colunas_select)}
    FROM weather_renomeado
"""

df = spark.sql(query_conversao_numerica)

df.createOrReplaceTempView("weather_convertido")

spark.sql("""
    SELECT *
    FROM weather_convertido
    LIMIT 5
""").show(truncate=False)

## 8. Criação das variáveis temporais

In [ ]:
df.createOrReplaceTempView("weather_convertido")

df = spark.sql("""
    SELECT
        *,
        CASE
            WHEN TO_DATE(data, 'yyyy-MM-dd') IS NOT NULL
                THEN TO_DATE(data, 'yyyy-MM-dd')
            ELSE TO_DATE(data, 'dd/MM/yyyy')
        END AS data_formatada
    FROM weather_convertido
""")

df.createOrReplaceTempView("weather_com_data")

df = spark.sql("""
    SELECT
        *,
        YEAR(data_formatada) AS ano,
        MONTH(data_formatada) AS mes,
        DAY(data_formatada) AS dia,
        HOUR(CAST(hora AS TIMESTAMP)) AS hora_num
    FROM weather_com_data
""")

df.createOrReplaceTempView("weather_temporal")

spark.sql("""
    SELECT
        data,
        hora,
        data_formatada,
        ano,
        mes,
        dia,
        hora_num
    FROM weather_temporal
    LIMIT 20
""").show(truncate=False)

## 9. Conferência do recorte de São Paulo/Visão geral da base

In [ ]:
df_sp = df

df_sp.createOrReplaceTempView("weather_sp_raw")

spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final
    FROM weather_sp_raw
""").show(truncate=False)

## 10. Tratamento de valores sentinela para EDA

In [ ]:
df_sp.createOrReplaceTempView("weather_sp_raw")

colunas_select = []

for coluna in df_sp.columns:
    if coluna in colunas_numericas:
        colunas_select.append(
            f"""
            CASE
                WHEN `{coluna}` <= -999 THEN NULL
                ELSE `{coluna}`
            END AS `{coluna}`
            """
        )
    else:
        colunas_select.append(f"`{coluna}`")

query_tratamento_sentinela = f"""
    SELECT
        {", ".join(colunas_select)}
    FROM weather_sp_raw
"""

df_sp_tratado = spark.sql(query_tratamento_sentinela)

df_sp_tratado.createOrReplaceTempView("weather_sp")

## 11. Dimensão da base de São Paulo

In [ ]:
spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT station_code) AS total_estacoes,
        COUNT(DISTINCT station) AS total_cidades_ou_estacoes,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final
    FROM weather_sp
""").show(truncate=False)

## 12. Qualidade dos dados: percentual de nulos

In [ ]:
df_sp_tratado.createOrReplaceTempView("weather_sp")

total_sp = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp
""").collect()[0]["total"]

expressoes_sql = []

for c in df_sp_tratado.columns:
    expressoes_sql.append(
        f"""
        ROUND(
            SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) / {total_sp} * 100,
            2
        ) AS `{c}`
        """
    )

query_nulos = f"""
    SELECT
        {",".join(expressoes_sql)}
    FROM weather_sp
"""

df_percentual_nulos = spark.sql(query_nulos)

df_percentual_nulos.show(truncate=False)

## 13. Cobertura temporal por ano

In [ ]:
registros_ano = spark.sql("""
    SELECT
        ano,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp
    WHERE ano IS NOT NULL
    GROUP BY ano
    ORDER BY ano
""")

registros_ano.show(100, truncate=False)

dados = registros_ano.collect()

anos = [linha["ano"] for linha in dados]
totais = [linha["total_registros"] for linha in dados]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=anos,
    y=totais,
    mode="lines+markers",
    name="Total de registros"
))

fig.update_layout(
    title="Quantidade de registros por ano - SP",
    xaxis_title="Ano",
    yaxis_title="Total de registros",
    template="plotly_white"
)

fig.show()

## 14. Lista de estações meteorológicas em São Paulo

In [ ]:
estacoes_sp = spark.sql("""
    SELECT DISTINCT
        station,
        station_code,
        latitude,
        longitude,
        altitude
    FROM weather_sp
    ORDER BY station
""")

estacoes_sp.show(300, truncate=False)

## 15. Classificação geográfica das estações

In [ ]:
weather_cols = [
    f"w.`{c}`"
    for c in df_sp_tratado.columns
    if c not in ["station", "latitude", "longitude", "altitude"]
]

weather_cols_sql = ",\n        ".join(weather_cols)

estacoes_geo = spark.sql("""
    SELECT
        station_code,
        FIRST(station, true) AS station,
        ROUND(AVG(latitude), 4) AS latitude,
        ROUND(AVG(longitude), 4) AS longitude,
        ROUND(AVG(altitude), 2) AS altitude,
        COUNT(*) AS total_registros_estacao
    FROM weather_sp
    WHERE station_code IS NOT NULL
    GROUP BY station_code
""")

estacoes_geo.createOrReplaceTempView("estacoes_geo")

estacoes_geo_classificada = spark.sql("""
    SELECT
        *,
        CASE
            WHEN altitude IS NULL THEN 'sem_info'
            WHEN altitude < 300 THEN 'baixa_altitude'
            WHEN altitude < 700 THEN 'media_altitude'
            ELSE 'alta_altitude'
        END AS faixa_altitude,

        CASE
            WHEN longitude <= -48.5 AND latitude <= -23.5 THEN 'sul_sudoeste'
            WHEN longitude <= -48.5 AND latitude > -23.5 THEN 'oeste_noroeste'
            WHEN longitude > -47.0 AND latitude <= -23.5 THEN 'leste_sudeste'
            WHEN longitude > -47.0 AND latitude > -23.5 THEN 'nordeste_vale'
            ELSE 'centro_metropolitana'
        END AS macro_regiao_sp,

        CASE
            WHEN altitude < 100 AND longitude > -47.0 THEN 'litoral'
            WHEN altitude >= 700 THEN 'serra_altitude'
            WHEN station RLIKE '(?i)SAO PAULO|SÃO PAULO|BARUERI|GUARULHOS|OSASCO|ABC|SANTO ANDRE|SANTO ANDRÉ|SAO BERNARDO|SÃO BERNARDO'
                THEN 'urbano_metropolitano'
            ELSE 'interior'
        END AS tipo_area
    FROM estacoes_geo
""")

estacoes_geo_classificada.createOrReplaceTempView("estacoes_geo_classificada")

df_sp_geo = spark.sql(f"""
    SELECT
        e.station,
        {weather_cols_sql},
        e.latitude,
        e.longitude,
        e.altitude,
        e.faixa_altitude,
        e.macro_regiao_sp,
        e.tipo_area
    FROM weather_sp w
    LEFT JOIN estacoes_geo_classificada e
        ON w.station_code = e.station_code
""")

df_sp_geo.createOrReplaceTempView("weather_sp_geo")

spark.sql("""
    SELECT
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp_geo
    GROUP BY macro_regiao_sp, tipo_area, faixa_altitude
    ORDER BY macro_regiao_sp, tipo_area, faixa_altitude
""").show(100, truncate=False)

## 16. Quantidade de estações por tipo de área

In [ ]:
estacoes_tipo_area = spark.sql("""
    SELECT
        tipo_area,
        COUNT(DISTINCT station_code) AS total_estacoes,
        COUNT(*) AS total_registros
    FROM weather_sp_geo
    GROUP BY tipo_area
    ORDER BY total_estacoes DESC
""")

estacoes_tipo_area.show(truncate=False)

estacoes_tipo_area_plot = [row.asDict() for row in estacoes_tipo_area.collect()]

fig = px.bar(
    estacoes_tipo_area_plot,
    x="tipo_area",
    y="total_estacoes",
    text="total_estacoes",
    title="Quantidade de estações por tipo de área - SP",
    labels={
        "tipo_area": "Tipo de área",
        "total_estacoes": "Total de estações"
    }
)

fig.update_traces(textposition="outside")
fig.show()

## 17. Temperatura média por tipo de área

In [ ]:
temp_tipo_area = spark.sql("""
    SELECT
        tipo_area,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        ROUND(STDDEV(temperatura), 2) AS desvio_temperatura,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
    GROUP BY tipo_area
    ORDER BY temperatura_media
""")

temp_tipo_area.show(truncate=False)

temp_tipo_area_plot = [row.asDict() for row in temp_tipo_area.collect()]

fig = px.bar(
    temp_tipo_area_plot,
    x="tipo_area",
    y="temperatura_media",
    text="temperatura_media",
    title="Temperatura média por tipo de área - SP",
    labels={
        "tipo_area": "Tipo de área",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 18. Temperatura média por macro região de São Paulo

In [ ]:
temp_macro = spark.sql("""
    SELECT
        macro_regiao_sp,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        ROUND(STDDEV(temperatura), 2) AS desvio_temperatura,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
    GROUP BY macro_regiao_sp
    ORDER BY temperatura_media
""")

temp_macro.show(truncate=False)

temp_macro_plot = [row.asDict() for row in temp_macro.collect()]

fig = px.bar(
    temp_macro_plot,
    x="macro_regiao_sp",
    y="temperatura_media",
    text="temperatura_media",
    title="Temperatura média por macro região - SP",
    labels={
        "macro_regiao_sp": "Macro região",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 19. Temperatura média por faixa de altitude

In [ ]:
temp_altitude = spark.sql("""
    SELECT
        faixa_altitude,
        ROUND(AVG(altitude), 2) AS altitude_media,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
    GROUP BY faixa_altitude
    ORDER BY altitude_media
""")

temp_altitude.show(truncate=False)

temp_altitude_plot = [row.asDict() for row in temp_altitude.collect()]

fig = px.bar(
    temp_altitude_plot,
    x="faixa_altitude",
    y="temperatura_media",
    text="temperatura_media",
    title="Temperatura média por faixa de altitude - SP",
    labels={
        "faixa_altitude": "Faixa de altitude",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 20. Ranking das estações mais frias e mais quentes

In [ ]:
estacoes_temp = spark.sql("""
    SELECT
        station,
        station_code,
        latitude,
        longitude,
        altitude,
        tipo_area,
        macro_regiao_sp,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
    GROUP BY station, station_code, latitude, longitude, altitude, tipo_area, macro_regiao_sp
    HAVING registros_validos > 1000
""")

estacoes_temp.createOrReplaceTempView("estacoes_temp")

print("Estações mais frias:")
spark.sql("""
    SELECT *
    FROM estacoes_temp
    ORDER BY temperatura_media ASC
    LIMIT 10
""").show(truncate=False)

print("Estações mais quentes:")
spark.sql("""
    SELECT *
    FROM estacoes_temp
    ORDER BY temperatura_media DESC
    LIMIT 10
""").show(truncate=False)

## 21. Umidade média por tipo de área

In [ ]:
umidade_tipo_area = spark.sql("""
    SELECT
        tipo_area,
        ROUND(AVG(umidade), 2) AS umidade_media,
        ROUND(MIN(umidade), 2) AS umidade_minima,
        ROUND(MAX(umidade), 2) AS umidade_maxima,
        COUNT(umidade) AS registros_validos
    FROM weather_sp_geo
    WHERE umidade IS NOT NULL
    GROUP BY tipo_area
    ORDER BY umidade_media DESC
""")

umidade_tipo_area.show(truncate=False)

umidade_tipo_area_plot = [row.asDict() for row in umidade_tipo_area.collect()]

fig = px.bar(
    umidade_tipo_area_plot,
    x="tipo_area",
    y="umidade_media",
    text="umidade_media",
    title="Umidade média por tipo de área - SP",
    labels={
        "tipo_area": "Tipo de área",
        "umidade_media": "Umidade média (%)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 22. Precipitação acumulada por tipo de área

In [ ]:
precipitacao_tipo_area = spark.sql("""
    SELECT
        tipo_area,
        ROUND(SUM(precipitacao), 2) AS precipitacao_total,
        ROUND(AVG(precipitacao), 4) AS precipitacao_media_horaria,
        COUNT(precipitacao) AS registros_validos
    FROM weather_sp_geo
    WHERE precipitacao IS NOT NULL
    GROUP BY tipo_area
    ORDER BY precipitacao_total DESC
""")

precipitacao_tipo_area.show(truncate=False)

precipitacao_tipo_area_plot = [row.asDict() for row in precipitacao_tipo_area.collect()]

fig = px.bar(
    precipitacao_tipo_area_plot,
    x="tipo_area",
    y="precipitacao_total",
    text="precipitacao_total",
    title="Precipitação acumulada por tipo de área - SP",
    labels={
        "tipo_area": "Tipo de área",
        "precipitacao_total": "Precipitação acumulada (mm)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 23. Sazonalidade mensal da temperatura

In [ ]:
temp_mes = spark.sql("""
    SELECT
        mes,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND mes IS NOT NULL
    GROUP BY mes
    ORDER BY mes
""")

temp_mes.show(12, truncate=False)

temp_mes_plot = [row.asDict() for row in temp_mes.collect()]

fig = px.line(
    temp_mes_plot,
    x="mes",
    y="temperatura_media",
    markers=True,
    title="Sazonalidade mensal da temperatura - SP",
    labels={
        "mes": "Mês",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 24. Sazonalidade mensal por tipo de área

In [ ]:
temp_mes_area = spark.sql("""
    SELECT
        mes,
        tipo_area,
        ROUND(AVG(temperatura), 2) AS temperatura_media
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND mes IS NOT NULL
    GROUP BY mes, tipo_area
    ORDER BY mes, tipo_area
""")

temp_mes_area.show(100, truncate=False)

temp_mes_area_plot = [row.asDict() for row in temp_mes_area.collect()]

fig = px.line(
    temp_mes_area_plot,
    x="mes",
    y="temperatura_media",
    color="tipo_area",
    markers=True,
    title="Temperatura média mensal por tipo de área - SP",
    labels={
        "mes": "Mês",
        "temperatura_media": "Temperatura média (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 25. Ciclo diário da temperatura

In [ ]:
temp_hora = spark.sql("""
    SELECT
        hora_num,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND hora_num IS NOT NULL
    GROUP BY hora_num
    ORDER BY hora_num
""")

temp_hora.show(24, truncate=False)

temp_hora_plot = [row.asDict() for row in temp_hora.collect()]

fig = px.line(
    temp_hora_plot,
    x="hora_num",
    y="temperatura_media",
    markers=True,
    title="Ciclo diário da temperatura - SP",
    labels={
        "hora_num": "Hora do dia",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 26. Ciclo diário por tipo de área

In [ ]:
temp_hora_area = spark.sql("""
    SELECT
        hora_num,
        tipo_area,
        ROUND(AVG(temperatura), 2) AS temperatura_media
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND hora_num IS NOT NULL
      AND tipo_area IS NOT NULL
    GROUP BY hora_num, tipo_area
    ORDER BY hora_num, tipo_area
""")

temp_hora_area.show(100, truncate=False)

temp_hora_area_plot = [row.asDict() for row in temp_hora_area.collect()]

fig = px.line(
    temp_hora_area_plot,
    x="hora_num",
    y="temperatura_media",
    color="tipo_area",
    markers=True,
    title="Ciclo diário médio da temperatura por tipo de área - SP",
    labels={
        "hora_num": "Hora do dia",
        "temperatura_media": "Temperatura média histórica (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 27. Tendência anual da temperatura em São Paulo

In [ ]:
temp_ano_sp = spark.sql("""
    SELECT
        ano,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(MIN(temperatura), 2) AS temperatura_minima,
        ROUND(MAX(temperatura), 2) AS temperatura_maxima,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND ano IS NOT NULL
    GROUP BY ano
    ORDER BY ano
""")

temp_ano_sp.show(100, truncate=False)

temp_ano_sp_plot = [row.asDict() for row in temp_ano_sp.collect()]

fig = px.line(
    temp_ano_sp_plot,
    x="ano",
    y="temperatura_media",
    markers=True,
    title="Temperatura média anual - Estado de São Paulo",
    labels={
        "ano": "Ano",
        "temperatura_media": "Temperatura média histórica (°C)"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 28. Tendência anual por tipo de área

In [ ]:
temp_ano_area = spark.sql("""
    SELECT
        ano,
        tipo_area,
        ROUND(AVG(temperatura), 2) AS temperatura_media
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND ano IS NOT NULL
      AND tipo_area IS NOT NULL
    GROUP BY ano, tipo_area
    ORDER BY ano, tipo_area
""")

temp_ano_area.show(200, truncate=False)

temp_ano_area_plot = [row.asDict() for row in temp_ano_area.collect()]

fig = px.line(
    temp_ano_area_plot,
    x="ano",
    y="temperatura_media",
    color="tipo_area",
    markers=True,
    title="Temperatura média anual por tipo de área - SP",
    labels={
        "ano": "Ano",
        "temperatura_media": "Temperatura média histórica (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 29. Relação entre altitude e temperatura

In [ ]:
altitude_temperatura = spark.sql("""
    SELECT
        station,
        station_code,
        altitude,
        tipo_area,
        macro_regiao_sp,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND altitude IS NOT NULL
    GROUP BY station, station_code, altitude, tipo_area, macro_regiao_sp
    HAVING registros_validos > 1000
    ORDER BY altitude
""")

altitude_temperatura.show(300, truncate=False)

altitude_temperatura_plot = [row.asDict() for row in altitude_temperatura.collect()]

fig = px.scatter(
    altitude_temperatura_plot,
    x="altitude",
    y="temperatura_media",
    color="tipo_area",
    hover_data=["station", "station_code", "macro_regiao_sp", "registros_validos"],
    title="Relação entre altitude e temperatura média - SP",
    labels={
        "altitude": "Altitude (m)",
        "temperatura_media": "Temperatura média (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.show()

## 30. Relação entre umidade e temperatura

In [ ]:
umidade_temperatura = spark.sql("""
    SELECT
        station,
        station_code,
        tipo_area,
        macro_regiao_sp,
        ROUND(AVG(umidade), 2) AS umidade_media,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        COUNT(temperatura) AS registros_validos
    FROM weather_sp_geo
    WHERE temperatura IS NOT NULL
      AND umidade IS NOT NULL
    GROUP BY station, station_code, tipo_area, macro_regiao_sp
    HAVING registros_validos > 1000
    ORDER BY umidade_media
""")

umidade_temperatura.show(300, truncate=False)

umidade_temperatura_plot = [row.asDict() for row in umidade_temperatura.collect()]

fig = px.scatter(
    umidade_temperatura_plot,
    x="umidade_media",
    y="temperatura_media",
    color="tipo_area",
    hover_data=["station", "station_code", "macro_regiao_sp", "registros_validos"],
    title="Relação entre umidade média e temperatura média - SP",
    labels={
        "umidade_media": "Umidade média (%)",
        "temperatura_media": "Temperatura média histórica (°C)",
        "tipo_area": "Tipo de área"
    }
)

fig.update_traces(marker=dict(size=9, opacity=0.75))
fig.show()

## 31. Áreas de interesse ambiental

In [ ]:
# Coordenadas aproximadas apenas para análise exploratória.

df_estacoes_interesse = spark.sql("""
    SELECT
        *,
        2 * 6371.0 * ASIN(
            SQRT(
                POWER(SIN((RADIANS(latitude) - RADIANS(-24.53)) / 2), 2) +
                COS(RADIANS(-24.53)) * COS(RADIANS(latitude)) *
                POWER(SIN((RADIANS(longitude) - RADIANS(-48.69)) / 2), 2)
            )
        ) AS dist_petar_km,

        2 * 6371.0 * ASIN(
            SQRT(
                POWER(SIN((RADIANS(latitude) - RADIANS(-23.24)) / 2), 2) +
                COS(RADIANS(-23.24)) * COS(RADIANS(latitude)) *
                POWER(SIN((RADIANS(longitude) - RADIANS(-46.95)) / 2), 2)
            )
        ) AS dist_serra_do_japi_km,

        2 * 6371.0 * ASIN(
            SQRT(
                POWER(SIN((RADIANS(latitude) - RADIANS(-24.15)) / 2), 2) +
                COS(RADIANS(-24.15)) * COS(RADIANS(latitude)) *
                POWER(SIN((RADIANS(longitude) - RADIANS(-48.05)) / 2), 2)
            )
        ) AS dist_serra_da_macaca_km
    FROM estacoes_geo_classificada
    WHERE latitude IS NOT NULL
      AND longitude IS NOT NULL
""")

df_estacoes_interesse.createOrReplaceTempView("estacoes_distancias")

df_estacoes_interesse = spark.sql("""
    SELECT
        *,
        CASE
            WHEN dist_petar_km <= dist_serra_do_japi_km
             AND dist_petar_km <= dist_serra_da_macaca_km
             AND dist_petar_km <= 80
                THEN 'PETAR'

            WHEN dist_serra_do_japi_km <= dist_petar_km
             AND dist_serra_do_japi_km <= dist_serra_da_macaca_km
             AND dist_serra_do_japi_km <= 80
                THEN 'Serra do Japi'

            WHEN dist_serra_da_macaca_km <= dist_petar_km
             AND dist_serra_da_macaca_km <= dist_serra_do_japi_km
             AND dist_serra_da_macaca_km <= 80
                THEN 'Serra da Macaca'

            ELSE 'Outra região'
        END AS area_interesse
    FROM estacoes_distancias
""")

df_estacoes_interesse.createOrReplaceTempView("estacoes_interesse")

spark.sql("""
    SELECT
        area_interesse,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM estacoes_interesse
    GROUP BY area_interesse
    ORDER BY total_estacoes DESC
""").show(truncate=False)

## 32. Temperatura e umidade nas áreas de interesse

In [ ]:
df_sp_geo.createOrReplaceTempView("weather_sp_geo")
df_estacoes_interesse.createOrReplaceTempView("estacoes_interesse")

df_sp_interesse = spark.sql("""
    SELECT
        w.*,
        e.area_interesse
    FROM weather_sp_geo w
    LEFT JOIN (
        SELECT
            station_code,
            area_interesse
        FROM estacoes_interesse
    ) e
        ON w.station_code = e.station_code
""")

df_sp_interesse.createOrReplaceTempView("weather_sp_interesse")

area_interesse_clima = spark.sql("""
    SELECT
        area_interesse,
        COUNT(DISTINCT station_code) AS total_estacoes,
        ROUND(AVG(temperatura), 2) AS temperatura_media,
        ROUND(AVG(umidade), 2) AS umidade_media,
        ROUND(SUM(precipitacao), 2) AS precipitacao_total
    FROM weather_sp_interesse
    WHERE area_interesse IS NOT NULL
    GROUP BY area_interesse
    ORDER BY temperatura_media
""")

area_interesse_clima.show(truncate=False)

area_interesse_clima_plot = [row.asDict() for row in area_interesse_clima.collect()]

fig = px.bar(
    area_interesse_clima_plot,
    x="area_interesse",
    y="temperatura_media",
    text="temperatura_media",
    title="Temperatura média por área de interesse - SP",
    labels={
        "area_interesse": "Área de interesse",
        "temperatura_media": "Temperatura média (°C)"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 33. Correlação com a temperatura

In [ ]:
variaveis_correlacao = [
    "latitude",
    "longitude",
    "altitude",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento",
    "mes",
    "hora_num"
]

variaveis_correlacao = [c for c in variaveis_correlacao if c in df_sp_geo.columns]

expressoes_corr = []

for c in variaveis_correlacao:
    expressoes_corr.append(
        f"""
        SELECT
            '{c}' AS variavel,
            ROUND(CORR({c}, temperatura), 4) AS correlacao_temperatura
        FROM weather_sp_geo
        WHERE {c} IS NOT NULL
          AND temperatura IS NOT NULL
        """
    )

query_correlacoes = """
    UNION ALL
""".join(expressoes_corr)

correlacoes_df = spark.sql(f"""
    SELECT
        variavel,
        correlacao_temperatura
    FROM (
        {query_correlacoes}
    )
    WHERE correlacao_temperatura IS NOT NULL
    ORDER BY ABS(correlacao_temperatura) DESC
""")

correlacoes_df.show(50, truncate=False)

## 34. Criação de base agregada diária para apoiar modelagem

In [ ]:
base_diaria_sp = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,
        ROUND(AVG(temperatura), 2) AS temp_media_dia,
        ROUND(MIN(temperatura), 2) AS temp_min_dia,
        ROUND(MAX(temperatura), 2) AS temp_max_dia,
        ROUND(AVG(umidade), 2) AS umidade_media_dia,
        ROUND(SUM(precipitacao), 2) AS precipitacao_total_dia,
        ROUND(AVG(pressao), 2) AS pressao_media_dia,
        ROUND(AVG(radiacao), 2) AS radiacao_media_dia,
        ROUND(AVG(velocidade_vento), 2) AS vento_medio_dia,
        COUNT(temperatura) AS medicoes_temp_validas
    FROM weather_sp_geo
    WHERE data_formatada IS NOT NULL
    GROUP BY
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude
    ORDER BY station_code, data_formatada
""")

base_diaria_sp.createOrReplaceTempView("base_diaria_sp")

base_diaria_sp.show(50, truncate=False)

## 35. Salvamento das bases de EDA

In [ ]:
eda_geo_path = "/home/jovyan/work/data/processed/weather_sp_eda_geo"
base_diaria_path = "/home/jovyan/work/data/processed/weather_sp_base_diaria"

df_sp_geo.write.mode("overwrite").parquet(eda_geo_path)
base_diaria_sp.write.mode("overwrite").parquet(base_diaria_path)

print(f"Base EDA geográfica salva em: {eda_geo_path}")
print(f"Base diária salva em: {base_diaria_path}")

## 36. Principais insights esperados para apresentação

Depois de executar este notebook, você pode organizar a apresentação em torno destas perguntas:

1. Quantas estações meteorológicas existem em São Paulo e qual o período analisado?
2. Quais variáveis possuem mais valores ausentes?
3. Quais regiões de SP apresentam maior ou menor temperatura média?
4. A altitude influencia a temperatura média?
5. Litoral, serra, interior e áreas urbanas apresentam diferenças claras?
6. A umidade é maior em regiões litorâneas ou de serra?
7. Como a temperatura muda ao longo dos meses?
8. Como a temperatura muda ao longo das horas do dia?
9. Quais estações são historicamente mais frias e mais quentes?
10. Que variáveis parecem mais relacionadas à temperatura?

Esses pontos conectam a EDA diretamente com a modelagem, pois mostram que a previsão de temperatura deve considerar:

- sazonalidade mensal;
- ciclo diário;
- altitude;
- latitude e longitude;
- umidade;
- pressão;
- precipitação;
- diferenças regionais dentro do estado.

## 37. Conclusão da EDA

Esta EDA mostrou que o dataset possui potencial para análises além da simples previsão de temperatura.

A base de São Paulo permite explorar diferenças espaciais importantes entre litoral, serra, interior e áreas urbanas.

A presença de latitude, longitude e altitude permite enriquecer a análise com uma dimensão geográfica.

Além disso, os padrões mensais e horários indicam forte componente temporal, justificando a criação de features de sazonalidade e bases específicas para previsão futura.

Próximos passos:

1. usar a base diária salva neste notebook;
2. criar o alvo `temperatura_amanha`;
3. criar o alvo `temperatura_media_proximos_7_dias`;
4. treinar e comparar Linear Regression, Random Forest e Redes Neurais.